# Phase 3 — PPO (RLHF étape 2)

Entraîne la policy via PPO en utilisant le DPO comme warm-start et le RM comme signal de récompense.

## ⚠️ Datasets à attacher (DEUX !)

Avant de lancer ce notebook : **Add Data** → ajoute les deux datasets produits aux phases 1 et 2 :
- `adl-dpo-adapter` (depuis le notebook 01)
- `adl-reward-model` (depuis le notebook 02)

Ajuste les paths `DPO_ZIP` et `RM_ZIP` ci-dessous selon les noms exacts de tes datasets.

**Setup Kaggle** : GPU T4 x1, Internet On.  
**Durée** : ~4-6 h pour 1 000 épisodes (125 steps gradient).


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Optionnel : pour la génération synthétique (data/generate_synthetic.py)
# from kaggle_secrets import UserSecretsClient
# os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")

In [ ]:
!pip install -q -U bitsandbytes transformers==4.46.3 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 datasets==3.2.0 sentence-transformers faiss-cpu \
    anthropic openai pyarrow==17.0.0 tqdm

In [ ]:
!rm -rf /kaggle/working/adl
!git clone https://github.com/FeelTheFloww/adl-ethics.git /kaggle/working/adl
%cd /kaggle/working/adl

In [ ]:
# ⚠️ Adapte ces chemins selon tes noms de datasets Kaggle
DPO_ZIP = "/kaggle/input/adl-dpo-adapter/dpo_model.zip"
RM_ZIP  = "/kaggle/input/adl-reward-model/reward_model.zip"

import os, zipfile, shutil
os.makedirs("results/dpo_model", exist_ok=True)
os.makedirs("results/reward_model", exist_ok=True)

with zipfile.ZipFile(DPO_ZIP) as z:
    z.extractall("results/dpo_model")
with zipfile.ZipFile(RM_ZIP) as z:
    z.extractall("results/reward_model")

print("DPO files:", os.listdir("results/dpo_model"))
print("RM files:",  os.listdir("results/reward_model"))

In [ ]:
!python data/prepare_preferences.py \
    --n_pku 15000 --n_ultra 5000 \
    --out_path data/preferences.jsonl

In [ ]:
!python training/train_ppo.py \
    --dpo_adapter results/dpo_model \
    --reward_model_path results/reward_model \
    --data_path data/preferences.jsonl \
    --output_dir results/rlhf_model

## Export

In [ ]:
import shutil, os
src = "/kaggle/working/adl/results/rlhf_model"
out = "/kaggle/working/rlhf_model.zip"
shutil.make_archive(out.replace(".zip", ""), "zip", src)
print(f"Zipped -> {out}  ({os.path.getsize(out)//1024//1024} MB)")
print("\nCréer Kaggle Dataset 'adl-rlhf-model' à attacher au notebook 04_eval")